In [54]:
import Pkg
Pkg.add("JuMP")
Pkg.add("HiGHS")
Pkg.add("MultiObjectiveAlgorithms")

   Resolving package versions...
     Project No packages added to or removed from `~/.julia/environments/v1.12/Project.toml`
    Manifest No packages added to or removed from `~/.julia/environments/v1.12/Manifest.toml`
Precompiling packages...
              ✗ CPLEX
  0 dependencies successfully precompiled in 2 seconds. 59 already precompiled.

The following 1 direct dependency failed to precompile:

CPLEX 

Failed to precompile CPLEX [a076750e-1247-5638-91d2-ce28b192dca0] to "/home/husted42/.julia/compiled/v1.12/CPLEX/jl_yNNnin".
ERROR: LoadError: CPLEX not properly installed. Please run Pkg.build("CPLEX")
Stacktrace:
  [1] error(s::String)
    @ Base ./error.jl:44
  [2] top-level scope
    @ ~/.julia/packages/CPLEX/5jmjD/src/CPLEX.jl:12
  [3] include(mod::Module, _path::String)
    @ Base ./Base.jl:306
  [4] include_package_for_output(pkg::Base.PkgId, input::String, depot_path::Vector{String}, dl_load_path::Vector{String}, load_path::Vector{String}, concrete_deps::Vector{Pair{Base.P

InterruptException: InterruptException:

In [3]:
Employees = ["A" "B" "C" "D" "E" "F" "G" "H" "I" "J" "K" "L"]  # list of employees
E=length(Employees)
Days = ["Mon" "Tue" "Wed" "Thur" "Fri" "Sat" "Sun"]
D=length(Days)
Hours= ["16-17" "17-18" "18-19" "19-20" "20-21" "21-22"]
H=length(Hours)

WorkerDemand=[ # WorkerDemand[day,hour]: no of needed workers
1 2 4 4 4 3;
1 2 2 4 4 3;
1 2 2 4 4 2;
2 2 3 3 4 3;
2 3 5 5 6 6;
4 5 5 6 6 6;
5 5 6 6 4 4 ]

Target = [8 8 8 10 10 10 15 15 20 20 20 20]

1×12 Matrix{Int64}:
 8  8  8  10  10  10  15  15  20  20  20  20

# Part 1

In [ ]:
using JuMP, HiGHS

########## ---------- Models ---------- ##########
model = Model(HiGHS.Optimizer)
set_optimizer_attribute(model, "log_to_console", false)

include("BurgerBarData.jl")



########## ---------- Variables ---------- ##########
# 1 if on a given day, hour, an employee are working
@variable(model, x[1:D, 1:H, 1:E], Bin)
# 1 if employee starts working at time h
@variable(model, y[1:D, 1:H, 1:E], Bin) 


########## ---------- Objectives ---------- ##########
@objective(model, Min,
    sum(y[d,h,e] for d in 1:D, h in 1:H, e in 1:E)
)

########## ---------- Constraints ---------- ##########
# The demand is exactly covered
@constraint(model, [d in 1:D, h in 1:H],
    sum(x[d,h,e] for e in 1:E) == WorkerDemand[d,h]
)

# Each Employee has to working within target hours in total
@constraint(model, [e in 1:E],
    sum(x[d,h,e] for d in 1:D, h in 1:H) >= Target[e] - 2
)
@constraint(model, [e in 1:E],
    sum(x[d,h,e] for d in 1:D, h in 1:H) <= Target[e] + 2
)


###### ------ Adding consecutivess hours ----- #####
# Can only start work once a day
@constraint(model, [d in 1:D, e in 1:E],
    sum(y[d, h, e] for h in 1:H) <= 1
)

# Can only work if he worked the previous hour or if just started working
@constraint(model, [d=1:D, h=1:H, e=1:E],
    x[d,h,e] <= (h > 1 ? x[d, h - 1, e] : 0) + y[d,h,e]
)

# Has to work 2 consecutive hours
@constraint(model, [e in 1:E, d in 1:D],
    sum(x[d, h, e] for h in 1:H) >= 2 * sum(y[d,h,e] for h in 1:H)
)


########## ---------- Optimize ---------- ##########
optimize!(model)

println("Optimal solution:")
println("z = ", (objective_value(model)))

println("Schedule")
for d in 1:D
    println("\n Day : ", d)
    println("Hours for a given Employee")
    for h in 1:H
        println(" h = ", h, value.(x[d, h, :]))
    end
    println("Starting working at")
    for h in 1:H
        println(value.(y[d, h, :]))
    end
end

println("Schedule")
for d in 1:D
    println("\n Day : ", d)
    for e in 1:E
        println(sum(value.(x[d, h, e]) for h in 1:H))
    end
end

println("no. of houes")
println(Target)
for e in 1:E
    print("\nE = ", e, " ")
    print(sum(value.(x[d,h,e]) for d in 1:D, h in 1:H), "t = ", Target[e])
end


println("no. of day in weekeend")
println(Target)
for e in 1:E
    print("\nE = ", e, " ")
    print(sum(value.(y[d,h,e]) for d in 6:7, h in 1:H), "t = ", 1)
end



Optimal solution:
z = 34.0
Schedule

 Day : 1
Hours for a given Employee
 h = 1[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0]
 h = 2[1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0]
 h = 3[1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 1.0]
 h = 4[1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 1.0]
 h = 5[1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 1.0]
 h = 6[1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0]
Starting working at
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0]
[1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0]
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]

 Day : 2
Hours for a given Employee
 h = 1[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0]
 h = 2[0.0, 0.0, 0.0, 0.0, 0.0, 0.0

# Part 2

In [ ]:
using JuMP, HiGHS

########## ---------- Models ---------- ##########
model = Model(HiGHS.Optimizer)
set_optimizer_attribute(model, "log_to_console", false)

include("BurgerBarData.jl")



########## ---------- Variables ---------- ##########
# 1 if on a given day, hour, an employee are working
@variable(model, x[1:D, 1:H, 1:E], Bin)
# 1 if employee starts working at time h
@variable(model, y[1:D, 1:H, 1:E], Bin) 


########## ---------- Objectives ---------- ##########
@objective(model, Min,
    sum(y[d,h,e] for d in 1:D, h in 1:H, e in 1:E)
)

########## ---------- Constraints ---------- ##########
# The demand is exactly covered
@constraint(model, [d in 1:D, h in 1:H],
    sum(x[d,h,e] for e in 1:E) == WorkerDemand[d,h]
)

# Each Employee has to working within target hours in total
@constraint(model, [e in 1:E],
    sum(x[d,h,e] for d in 1:D, h in 1:H) >= Target[e] - 2
)
@constraint(model, [e in 1:E],
    sum(x[d,h,e] for d in 1:D, h in 1:H) <= Target[e] + 2
)


###### ------ Constraints : Adding consecutivess hours ----- #####
# Can only start work once a day
@constraint(model, [d in 1:D, e in 1:E],
    sum(y[d, h, e] for h in 1:H) <= 1
)

# Can only work if he worked the previous hour or if just started working
@constraint(model, [d=1:D, h=1:H, e=1:E],
    x[d,h,e] <= (h > 1 ? x[d, h - 1, e] : 0) + y[d,h,e]
)

# Has to work 2 consecutive hours
@constraint(model, [e in 1:E, d in 1:D],
    sum(x[d, h, e] for h in 1:H) >= 2 * sum(y[d,h,e] for h in 1:H)
)

###### ------ Constraints : Each employee must only work 1 day in the weekeend ----- #####
# Ecah student most at most work 1 day in the weekeend
@constraint(model, [e in 1:E],
    sum(y[6,h,e]  for h in 1:H) + sum(y[7,h,e]  for h in 1:H) <= 1
)

########## ---------- Optimize ---------- ##########
optimize!(model)

println("Optimal solution:")
println("z = ", (objective_value(model)))

println("Schedule")
for d in 1:D
    println("\n Day : ", d)
    println("Hours for a given Employee")
    for h in 1:H
        println(" h = ", h, value.(x[d, h, :]))
    end
    println("Starting working at")
    for h in 1:H
        println(value.(y[d, h, :]))
    end
end

println("Schedule")
for d in 1:D
    println("\n Day : ", d)
    for e in 1:E
        println(sum(value.(x[d, h, e]) for h in 1:H))
    end
end

println("no. of houes")
println(Target)
for e in 1:E
    print("\nE = ", e, " ")
    print(sum(value.(x[d,h,e]) for d in 1:D, h in 1:H), "t = ", Target[e])
end

println("no. of day in weekeend")
println(Target)
for e in 1:E
    print("\nE = ", e, " ")
    print(sum(value.(y[d,h,e]) for d in 6:7, h in 1:H), "t = ", 1)
end




Optimal solution:
z = 34.0
Schedule

 Day : 1
Hours for a given Employee
 h = 1[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0]
 h = 2[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0]
 h = 3[1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 1.0, 0.0, 0.0, 0.0]
 h = 4[1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 1.0, 0.0, 0.0, 0.0]
 h = 5[1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 1.0, 0.0, 0.0, 0.0]
 h = 6[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 1.0, 0.0, 0.0, 0.0]
Starting working at
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0]
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0]
[1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0]
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]

 Day : 2
Hours for a given Employee
 h = 1[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0]
 h = 2[0.0, 0.0, 0.0, 0.0, 0.0, 0.0

In [ ]:
using MultiObjectiveAlgorithms
# Objective 1: Maximize Profit
@expression(MO_IC, Profit_expr, sum(Profit[c] * x[c] for c=1:C))
# Objective 2: Maximize Rating
@expression(MO_IC, Rating_expr, sum(Rating[c] * x[c] for c=1:C))
# Define combined BI-OBJECTIVE
@objective(MO_IC, Max, [Profit_expr, Rating_expr])

In [ ]:
Pkg.add("MultiObjectiveAlgorithms") # Forgot to download this on my laptop :(
using JuMP, HiGHS, MultiObjectiveAlgorithms

########## ---------- Models ---------- ##########
model = Model()
# include("BurgerBarData.jl")

########## ---------- Variables ---------- ##########
# 1 if on a given day, hour, an employee are working
@variable(model, x[1:D, 1:H, 1:E], Bin)
# 1 if employee starts working at time h
@variable(model, y[1:D, 1:H, 1:E], Bin) 


########## ---------- Objectives ---------- ##########
@expression(model, workhours_dev, 
    sum(sum(x[d,h,e] for d in 1:D, h in 1:H) - Target[e] for e in 1:E)
)
@expression(model, workdays_tot, 
    sum(y[d,h,e] for d in 1:D, h in 1:H, e in 1:E)
)

@objective(model, Min, [workhours_dev, workdays_tot])


########## ---------- Constraints ---------- ##########
# The demand is exactly covered
@constraint(model, [d in 1:D, h in 1:H],
    sum(x[d,h,e] for e in 1:E) == WorkerDemand[d,h]
)

# Each Employee has to working within target hours in total
@constraint(model, [e in 1:E],
    sum(x[d,h,e] for d in 1:D, h in 1:H) >= Target[e] - 2
)
@constraint(model, [e in 1:E],
    sum(x[d,h,e] for d in 1:D, h in 1:H) <= Target[e] + 2
)


###### ------ Constraints : Adding consecutivess hours ----- #####
# Can only start work once a day
@constraint(model, [d in 1:D, e in 1:E],
    sum(y[d, h, e] for h in 1:H) <= 1
)

# Can only work if he worked the previous hour or if just started working
@constraint(model, [d=1:D, h=1:H, e=1:E],
    x[d,h,e] <= (h > 1 ? x[d, h - 1, e] : 0) + y[d,h,e]
)

# Has to work 2 consecutive hours
@constraint(model, [e in 1:E, d in 1:D],
    sum(x[d, h, e] for h in 1:H) >= 2 * sum(y[d,h,e] for h in 1:H)
)

###### ------ Constraints : Each employee must only work 1 day in the weekeend ----- #####
# Ecah student most at most work 1 day in the weekeend
@constraint(model, [e in 1:E],
    sum(y[6,h,e]  for h in 1:H) + sum(y[7,h,e]  for h in 1:H) <= 1
)

########## ---------- Optimize ---------- ##########
# Set MIP solver
set_optimizer(model, () -> MultiObjectiveAlgorithms.Optimizer(HiGHS.Optimizer))
set_silent(model)
# Set MO solver
    set_attribute(model, MultiObjectiveAlgorithms.Algorithm(),
MultiObjectiveAlgorithms.EpsilonConstraint())
optimize!(model)
solution_summary(model)

ArgumentError: ArgumentError: Package MultiObjectiveAlgorithms not found in current path.
- Run `import Pkg; Pkg.add("MultiObjectiveAlgorithms")` to install the MultiObjectiveAlgorithms package.